# 04 — Inference

Runs the treated-unit inferential battery from [validation.md §5e](../docs/validation.md):

1. **In-space placebo** — refit treating each donor as treated; compute Brent's permutation p-value
2. **In-time placebo** — refit with $T^{\text{fake}}_0$ = 6 months before real $T_0$
3. **Leave-one-donor-out** — drop each donor with weight > 0.05, recompute the gap, report stability

**Inputs**: requires `02_Fit_Models` to have run (so fits exist in `data/results/`).  
**Outputs**: `data/validation/inference_{test}_{event}_{model}.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, T0_FAKE, MODEL_HPARAMS, DONOR_POOL_VARIANT, LOO_MIN_WEIGHT
from lib.data import build_panel, load_fit, save_validation_table
from lib.validation import in_space_placebo, in_time_placebo, leave_one_out, gap_distribution

EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'
VARIANT = DONOR_POOL_VARIANT
MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bsts']

print(f'Inferential battery: {EVENTS} × {MODELS}')

Inferential battery: ['russia', 'hormuz'] × ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bsts']
Last run: 2026-05-25 20:36:48


## §5e (i) — In-space placebo

For each model: refit treating each donor as the placebo treated unit; compute the post/pre RMSPE ratio. Brent's rank in the placebo distribution gives the permutation p-value (< 0.10 is the Abadie convention).

*(Note: this is computationally heavy — N_donors × N_models × N_events fits. For convex SCM each placebo takes ~3s; for XGBoost/Bayesian Ridge much faster. Total runtime ~5-10 min.)*

In [2]:
import time
from lib.validation import get_tuned_hparams

iso_results = {}
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        # Use the val-tuned hyperparameters from 02_Fit_Models so placebo runs match the headline fit.
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40   # cut down for placebo speed
        t0 = time.time()
        try:
            df = in_space_placebo(model, panel, 'Brent', meta['donors'],
                                  t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            iso_results[(event, model)] = df
            save_validation_table(df, f'inference_inspace_{event}_{model}')
            brent_p = float(df.loc['Brent', 'p_value']) if 'Brent' in df.index else np.nan
            print(f'  {event:6s} / {model:12s}  Brent p = {brent_p:.3f}  ({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'  {event:6s} / {model:12s}  ERROR: {str(e)[:60]}')

  russia / convex_scm    Brent p = 0.545  (14s)


  russia / ascm          Brent p = 0.682  (14s)
  russia / elastic_net   Brent p = 0.364  (0s)


  russia / xgboost       Brent p = 0.091  (5s)
  russia / bsts          Brent p = 0.636  (0s)


  hormuz / convex_scm    Brent p = 0.045  (16s)


  hormuz / ascm          Brent p = 0.045  (16s)
  hormuz / elastic_net   Brent p = 0.045  (0s)


  hormuz / xgboost       Brent p = 0.045  (5s)
  hormuz / bsts          Brent p = 0.045  (0s)
Last run: 2026-05-25 20:37:57


In [3]:
# Summary: Brent's p-value across models × events
rows = []
for (event, model), df in iso_results.items():
    if 'Brent' not in df.index:
        continue
    row = df.loc['Brent'].to_dict()
    row.update({'event': event, 'model': model})
    rows.append(row)

iso_brent = pd.DataFrame(rows)
if len(iso_brent):
    save_validation_table(iso_brent, 'inference_inspace_brent_summary')
    iso_brent.round(4)
else:
    print('No in-space placebo results to summarize.')

Last run: 2026-05-25 20:37:57


## §5e (i) — In-space placebo, **33-donor full pool**

The shared 21-donor pool has a permutation floor at $1/22 \approx 0.045$ — every Hormuz Brent p-value above sits exactly at this floor and cannot be distinguished from the saturation value. Re-running under the 33-donor full pool (per [validation.md §5e (i)](../docs/validation.md)) drops the floor to $1/34 \approx 0.029$ and gives resolution below conventional significance thresholds.

For Russia, the 33-donor pool also includes the 12 audit-flagged contaminated donors (Wheat, Corn, Palladium, EUR, DXY, BTC, Copper, IronOre, Soybeans, EM_Eq, GBP, US10Y) — Brent still ranking high there is a strictly more conservative test than under the clean 21-donor pool, since those donors' own post-period RMSPE was inflated by the same Russia shock and pushes the placebo distribution upward.

Outputs: `data/validation/inference_inspace_{event}_{model}_full.csv`. Comparison: `inference_inspace_brent_pool_comparison.csv`.

In [4]:
import time
from lib.validation import get_tuned_hparams

iso_full_results = {}
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant='full')
    print(f'\n=== {event}  /  variant=full  ({len(meta["donors"])} donors) ===')
    for model in MODELS:
        try:
            tuned = get_tuned_hparams(model, event, WINDOW, 'full')
        except Exception:
            tuned = get_tuned_hparams(model, event, WINDOW, 'shared')
        kwargs = {**MODEL_HPARAMS.get(model, {}), **tuned}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40
        t0 = time.time()
        try:
            df = in_space_placebo(model, panel, 'Brent', meta['donors'],
                                  t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            iso_full_results[(event, model)] = df
            save_validation_table(df, f'inference_inspace_{event}_{model}_full')
            brent_p = float(df.loc['Brent', 'p_value']) if 'Brent' in df.index else float('nan')
            print(f'  {model:12s}  Brent p = {brent_p:.4f}  ({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'  {model:12s}  ERROR: {str(e)[:60]}')


=== russia  /  variant=full  (33 donors) ===


  convex_scm    Brent p = 0.5000  (44s)


  ascm          Brent p = 0.4706  (44s)
  elastic_net   Brent p = 0.4118  (0s)


  xgboost       Brent p = 0.0588  (2s)
  bsts          Brent p = 0.5294  (0s)

=== hormuz  /  variant=full  (33 donors) ===


  convex_scm    Brent p = 0.0294  (50s)


  ascm          Brent p = 0.0294  (49s)
  elastic_net   Brent p = 0.0294  (0s)


  xgboost       Brent p = 0.0294  (2s)
  bsts          Brent p = 0.0294  (0s)
Last run: 2026-05-25 20:41:08


In [5]:
# Side-by-side 21- vs 33-donor in-space placebo p-values for Brent
comparison_rows = []
for event in EVENTS:
    for model in MODELS:
        row = {'event': event, 'model': model}
        for pool_name, suffix in [('p_21donor', ''), ('p_33donor', '_full')]:
            path = ROOT / 'data' / 'validation' / f'inference_inspace_{event}_{model}{suffix}.csv'
            if not path.exists():
                row[pool_name] = float('nan')
                continue
            df = pd.read_csv(path, index_col=0)
            if 'Brent' not in df.index:
                continue
            n_units = int(df['ratio'].notna().sum()) if 'ratio' in df.columns else len(df)
            row[pool_name] = float(df.loc['Brent', 'p_value'])
            row[f'floor_{pool_name[-7:]}'] = round(1.0 / n_units, 4)
        comparison_rows.append(row)

pool_compare = pd.DataFrame(comparison_rows)
save_validation_table(pool_compare, 'inference_inspace_brent_pool_comparison')
print('Brent in-space placebo p-value — 21- vs 33-donor pools:\n')
print(pool_compare.round(4).to_string(index=False))

Brent in-space placebo p-value — 21- vs 33-donor pools:

 event       model  p_21donor  floor_21donor  p_33donor  floor_33donor
russia  convex_scm     0.5455         0.0455     0.5000         0.0294
russia        ascm     0.6818         0.0455     0.4706         0.0294
russia elastic_net     0.3636         0.0455     0.4118         0.0294
russia     xgboost     0.0909         0.0455     0.0588         0.0294
russia        bsts     0.6364         0.0455     0.5294         0.0294
hormuz  convex_scm     0.0455         0.0455     0.0294         0.0294
hormuz        ascm     0.0455         0.0455     0.0294         0.0294
hormuz elastic_net     0.0455         0.0455     0.0294         0.0294
hormuz     xgboost     0.0455         0.0455     0.0294         0.0294
hormuz        bsts     0.0455         0.0455     0.0294         0.0294
Last run: 2026-05-25 20:41:08


## §5e (ii) — In-time placebo

Refit with fake $T_0$ = 6 months before the real event. The gap in the fake post-period (from $T^{\text{fake}}_0$ to real $T_0$) should be small — large gap means the SCM is finding spurious effects.

In [6]:
from lib.validation import get_tuned_hparams

intime_rows = []
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    t0_fake = T0_FAKE[event]
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = in_time_placebo(model, panel, 'Brent', meta['donors'],
                                t0_fake=t0_fake, t_pre_start=meta['t_pre_start'], **kwargs)
            # Compute gap in the fake post-period (t0_fake to real t0)
            fake_post = r['gap'][(r['gap'].index >= t0_fake) & (r['gap'].index < meta['t0'])]
            mean_fake_gap_pct = float(100 * (np.exp(fake_post.mean()) - 1)) if len(fake_post) > 0 else np.nan
            intime_rows.append({
                'event': event, 'model': model,
                't0_fake': str(t0_fake.date()),
                'fake_post_obs': len(fake_post),
                'mean_fake_gap_pct': mean_fake_gap_pct,
                'pre_rmspe_log': r['rmspe_pre'],
            })
        except Exception as e:
            intime_rows.append({'event': event, 'model': model, 'error': str(e)[:60]})

intime_df = pd.DataFrame(intime_rows)
save_validation_table(intime_df, 'inference_intime')
intime_df.round(4)

,event,model,t0_fake,fake_post_obs,mean_fake_gap_pct,pre_rmspe_log
0,russia,convex_scm,2021-08-24,129,1.8527,0.1180
1,russia,ascm,2021-08-24,129,-4.2503,0.1180
2,russia,elastic_net,2021-08-24,129,-2.8031,0.0565
3,russia,xgboost,2021-08-24,129,17.2925,0.0380
4,russia,bsts,2021-08-24,129,-5.3361,0.0315
5,hormuz,convex_scm,2025-08-01,127,-11.9101,0.0586
6,hormuz,ascm,2025-08-01,127,-12.3545,0.0586
7,hormuz,elastic_net,2025-08-01,127,-3.3215,0.0438
8,hormuz,xgboost,2025-08-01,127,-8.3097,0.0476
9,hormuz,bsts,2025-08-01,127,-12.1660,0.0310


Last run: 2026-05-25 20:41:14


## §5e (iii) — Leave-one-donor-out

Drop each high-weight donor and recompute the gap. Stability (small range around the baseline gap) means no single donor is driving the headline.

In [7]:
from lib.validation import get_tuned_hparams

for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40
        try:
            loo_results = leave_one_out(model, panel, 'Brent', meta['donors'],
                                        t0=meta['t0'], t_pre_start=meta['t_pre_start'],
                                        min_weight=LOO_MIN_WEIGHT, **kwargs)
            dist = gap_distribution(loo_results, t0=meta['t0'])
            save_validation_table(dist, f'inference_loo_{event}_{model}')
            print(f'\n{event} / {model} (baseline + {len(dist)-1} leave-outs):')
            print(dist.round(3).to_string())
        except Exception as e:
            print(f'{event} / {model}  ERROR: {str(e)[:80]}')


russia / convex_scm (baseline + 3 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        29.303          27.840       60.682        7.182
Coffee           28.804          25.832       69.874        3.445
Sugar            27.111          24.199       62.261        5.184
Cotton           33.685          37.599       59.309        3.431



russia / ascm (baseline + 17 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        13.362          12.673       51.173       -9.216
Silver           12.966          11.982       51.587       -9.236
Platinum          6.248           4.805       56.659      -15.941
Gold             11.768          10.418       48.624       -9.392
Coffee           14.794          11.469       50.533       -7.244
Cotton           10.788          15.191       49.697      -25.550
SP500            13.953          13.399       50.977       -8.499
Nikkei           11.949          11.275       52.838      -11.322
JPY              17.266          16.360       51.502       -4.092
CHF              13.083          12.696       51.522      -10.039
CNY              13.959          13.612       50.860       -8.379
INR              13.705          13.063       51.368       -8.912
KRW              15.823          


russia / xgboost (baseline + 5 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        36.868          36.652       88.349        9.158
Sugar            36.003          36.259       88.322        7.910
Cotton           35.273          35.106       97.566        9.297
SP500            44.595          42.352       99.839        9.402
WorldEq          36.001          35.264       97.200        8.738
JPY              42.176          41.278      101.889        8.867

russia / bsts (baseline + 18 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline          0.504           3.765       49.924      -31.409
Platinum          -5.624          -4.816       57.067      -34.886
Gold               3.110           4.085       45.132      -23.462
Coffee            -2.572           1.375       47


hormuz / convex_scm (baseline + 4 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        39.291          50.002      103.276        1.814
Sugar            30.331          37.153       83.381        0.103
JPY              39.690          50.201      100.826        4.051
KRW              38.556          49.329      102.624        1.382
TLT              38.381          48.085       99.606        2.071



hormuz / ascm (baseline + 16 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         42.956          53.173      107.641        2.382
Platinum          41.409          51.997      105.326        1.428
Gold              40.857          51.846      104.941        2.504
Coffee            46.007          56.099      112.605        3.236
Cotton            41.301          51.579      102.125        2.782
LiveCattle        45.424          56.318      112.971        3.230
SP500             43.114          53.231      107.693        2.497
WorldEq           42.468          51.878      105.786        2.137
Nikkei            42.797          53.848      108.134        2.636
AUD               48.533          61.346      115.855        5.108
JPY               41.941          52.766      105.874        2.261
CHF               49.601          63.735      120.250        2.941
CNY               4


hormuz / xgboost (baseline + 7 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        37.805          48.659      103.347        0.100
Silver           37.797          48.651      103.335        0.095
Gold             35.995          46.366      100.210       -0.887
Coffee           41.033          51.881      107.753        2.311
JPY              37.663          48.506      103.137       -0.000
CHF              37.269          47.747      102.100       -0.004
INR              37.380          47.878      102.279        0.068
MXN              36.905          47.354      101.561        0.239

hormuz / bsts (baseline + 18 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         43.607          55.969      113.116        2.346
Platinum          42.897          54.839      112.4